# Advanced Biophysical Modeling

This notebook demonstrates the advanced biophysical modeling capabilities of neuros-mechint:
- Ion channel dynamics (Hodgkin-Huxley formalism)
- Multi-compartment neurons with cable theory
- Synaptic plasticity (STDP, STP, homeostatic)
- Metabolic constraints (ATP dynamics, energy budgets)

These models enable biologically realistic simulations for foundation model testing.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from neuros_mechint.biophysical import (
    SodiumChannel, PotassiumChannel, CalciumChannel,
    MultiCompartmentNeuron, PrefabNeurons,
    STDP, ShortTermPlasticity,
    ATPDynamics, MetabolicConstraint
)

%matplotlib inline

## 1. Ion Channel Dynamics

Model voltage-gated ion channels using Hodgkin-Huxley formalism.

In [ ]:
# Create sodium and potassium channels
na_channel = SodiumChannel(g_max=120.0)  # mS/cm²
k_channel = PotassiumChannel(g_max=36.0)   # mS/cm²

# Voltage clamp protocol
voltages = torch.linspace(-80, 40, 100)
dt = 0.1  # ms

# Simulate steady-state activation
na_currents = []
k_currents = []

for V in voltages:
    # Simulate until steady state (100 ms)
    for _ in range(1000):
        na_channel.step(V, dt)
        k_channel.step(V, dt)
    
    na_currents.append(na_channel.current(V))
    k_currents.append(k_channel.current(V))

# Plot I-V curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(voltages.numpy(), na_currents, label='Na+', linewidth=2)
plt.plot(voltages.numpy(), k_currents, label='K+', linewidth=2)
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (μA/cm²)')
plt.title('Ion Channel I-V Curves')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot activation curves
plt.subplot(1, 2, 2)
na_activation = [na_channel.m.item() for _ in voltages]
k_activation = [k_channel.n.item() for _ in voltages]
plt.plot(voltages.numpy(), na_activation, label='Na+ activation (m)', linewidth=2)
plt.plot(voltages.numpy(), k_activation, label='K+ activation (n)', linewidth=2)
plt.xlabel('Voltage (mV)')
plt.ylabel('Activation')
plt.title('Gating Variable Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Multi-Compartment Neurons

Simulate neurons with dendrites, soma, and axon using cable theory.

In [ ]:
# Create a pyramidal cell
neuron = PrefabNeurons.pyramidal_cell(soma_area=1000.0)

print(f"Neuron has {len(neuron.compartments)} compartments:")
for comp in neuron.compartments:
    print(f"  {comp.name}: {comp.area:.1f} μm²")

# Current injection protocol
dt = 0.1  # ms
duration = 200  # ms
n_steps = int(duration / dt)

# Create step current (inject into soma)
current = torch.zeros(n_steps)
current[500:1500] = 500.0  # 500 pA for 100 ms

# Simulate
voltages = neuron.simulate(current, n_steps, dt)

# Plot results
time = np.arange(n_steps) * dt

plt.figure(figsize=(12, 8))

# Current injection
plt.subplot(3, 1, 1)
plt.plot(time, current.numpy(), 'k', linewidth=2)
plt.ylabel('Current (pA)')
plt.title('Current Injection Protocol')
plt.grid(True, alpha=0.3)

# Voltage traces for each compartment
plt.subplot(3, 1, 2)
for i, comp in enumerate(neuron.compartments):
    plt.plot(time, voltages[:, i].numpy(), label=comp.name, linewidth=2)
plt.ylabel('Voltage (mV)')
plt.title('Compartment Voltages')
plt.legend()
plt.grid(True, alpha=0.3)

# Soma voltage detail
plt.subplot(3, 1, 3)
soma_idx = 0  # Soma is first compartment
plt.plot(time, voltages[:, soma_idx].numpy(), 'r', linewidth=2)
plt.ylabel('Soma Voltage (mV)')
plt.xlabel('Time (ms)')
plt.title('Soma Response (Action Potentials)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Count spikes
spike_threshold = -20.0
soma_v = voltages[:, soma_idx].numpy()
spikes = np.where((soma_v[:-1] < spike_threshold) & (soma_v[1:] >= spike_threshold))[0]
print(f"\nGenerated {len(spikes)} action potentials")
print(f"Spike rate: {len(spikes) / (duration / 1000):.1f} Hz")

## 3. Synaptic Plasticity

Model spike-timing-dependent plasticity (STDP) and short-term plasticity (STP).

In [ ]:
# Create STDP model
stdp = STDP(tau_plus=20.0, tau_minus=20.0, A_plus=0.01, A_minus=0.01)

# STDP curve: weight change vs spike timing
delta_t_range = np.linspace(-100, 100, 200)  # ms
weight_changes = []

for delta_t in delta_t_range:
    if delta_t > 0:  # Pre before post (LTP)
        dw = stdp.A_plus * np.exp(-delta_t / stdp.tau_plus)
    else:  # Post before pre (LTD)
        dw = -stdp.A_minus * np.exp(delta_t / stdp.tau_minus)
    weight_changes.append(dw)

plt.figure(figsize=(12, 4))

# STDP window
plt.subplot(1, 2, 1)
plt.plot(delta_t_range, weight_changes, 'b', linewidth=2)
plt.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.axvline(0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Δt = t_post - t_pre (ms)')
plt.ylabel('Δw (weight change)')
plt.title('STDP Learning Window')
plt.grid(True, alpha=0.3)
plt.text(30, 0.008, 'LTP\n(potentiation)', fontsize=10, ha='center')
plt.text(-30, -0.008, 'LTD\n(depression)', fontsize=10, ha='center')

# Short-term plasticity
stp = ShortTermPlasticity(U=0.5, tau_rec=800.0, tau_fac=0.0)  # Depressing synapse

# Spike train (10 Hz)
spike_times = np.arange(0, 1000, 100)  # ms
dt = 1.0  # ms
n_steps = 1100

responses = []
for t in range(n_steps):
    if t in spike_times:
        response = stp.modulate(1.0)
        responses.append(response)
    stp.update(dt)

plt.subplot(1, 2, 2)
plt.plot(spike_times, responses, 'o-', markersize=8, linewidth=2)
plt.xlabel('Time (ms)')
plt.ylabel('Synaptic Response')
plt.title('Short-Term Depression (10 Hz train)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Metabolic Constraints

Model ATP dynamics and energy budgets to understand metabolic limitations.

In [ ]:
# Create ATP dynamics model
atp = ATPDynamics(
    ATP_0=2.5,          # mM (baseline)
    production_rate=1.0, # mM/s
    basal_consumption=0.1, # mM/s
    spike_cost=0.05     # mM per spike
)

# Simulate neural activity with varying spike rates
duration = 10000  # ms
dt = 1.0  # ms
n_steps = int(duration / dt)

# Create spike train with varying frequency
time = np.arange(n_steps) * dt / 1000  # seconds
spike_rate = 5 + 20 * np.sin(2 * np.pi * time / 2)  # Hz, oscillating 5-25 Hz

atp_levels = []
spike_counts = []

for t in range(n_steps):
    # Generate Poisson spikes
    spike_prob = spike_rate[t] * dt / 1000
    spike_occurred = np.random.rand() < spike_prob
    
    # Update ATP
    atp.update(spike_occurred, dt / 1000)
    
    atp_levels.append(atp.ATP)
    spike_counts.append(1 if spike_occurred else 0)

# Plot results
plt.figure(figsize=(12, 8))

# Spike rate
plt.subplot(3, 1, 1)
plt.plot(time, spike_rate, 'k', linewidth=2)
plt.ylabel('Spike Rate (Hz)')
plt.title('Neural Activity Pattern')
plt.grid(True, alpha=0.3)

# ATP levels
plt.subplot(3, 1, 2)
plt.plot(time, atp_levels, 'b', linewidth=2)
plt.axhline(atp.ATP_0, color='r', linestyle='--', label='Baseline', alpha=0.5)
plt.ylabel('[ATP] (mM)')
plt.title('ATP Dynamics')
plt.legend()
plt.grid(True, alpha=0.3)

# Running spike count
plt.subplot(3, 1, 3)
window_size = 1000  # 1 second
running_rate = np.convolve(spike_counts, np.ones(window_size) / window_size, mode='same')
plt.plot(time, running_rate * 1000, 'g', linewidth=2)  # Convert to Hz
plt.ylabel('Instantaneous Rate (Hz)')
plt.xlabel('Time (s)')
plt.title('Realized Spike Rate')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate metabolic metrics
mean_atp = np.mean(atp_levels)
min_atp = np.min(atp_levels)
total_spikes = np.sum(spike_counts)
energy_per_spike = atp.spike_cost

print(f"\nMetabolic Statistics:")
print(f"Mean ATP level: {mean_atp:.3f} mM")
print(f"Minimum ATP level: {min_atp:.3f} mM")
print(f"Total spikes: {total_spikes}")
print(f"Total energy consumed: {total_spikes * energy_per_spike:.2f} mM·ms")
print(f"ATP depletion: {(1 - min_atp / atp.ATP_0) * 100:.1f}%")

## 5. Integration with Foundation Models

Use biophysical constraints to test foundation model predictions.

In [ ]:
# Create metabolic constraint for foundation model testing
constraint = MetabolicConstraint(
    max_spike_rate=50.0,    # Hz
    energy_budget=1000.0,    # arbitrary units
    cost_per_spike=1.0
)

# Simulate foundation model activations
# (In practice, these would come from your model)
model_activations = torch.randn(100, 512)  # 100 samples, 512 neurons

# Convert to spike rates (threshold + normalize)
threshold = model_activations.mean()
spike_rates = torch.relu(model_activations - threshold).mean(dim=0) * 50  # Scale to 0-50 Hz

# Check metabolic feasibility
is_feasible = constraint.check_feasibility(spike_rates)
energy_consumption = constraint.compute_energy_cost(spike_rates)

print(f"Model predictions metabolically feasible: {is_feasible}")
print(f"Energy consumption: {energy_consumption:.1f} / {constraint.energy_budget:.1f}")
print(f"Budget usage: {energy_consumption / constraint.energy_budget * 100:.1f}%")

# Visualize
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(spike_rates.numpy(), bins=30, edgecolor='black')
plt.axvline(constraint.max_spike_rate, color='r', linestyle='--', 
            label=f'Max rate ({constraint.max_spike_rate} Hz)', linewidth=2)
plt.xlabel('Predicted Spike Rate (Hz)')
plt.ylabel('Count')
plt.title('Foundation Model Spike Rate Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
cumulative_cost = torch.cumsum(torch.sort(spike_rates, descending=True)[0], dim=0).numpy()
plt.plot(cumulative_cost, linewidth=2)
plt.axhline(constraint.energy_budget, color='r', linestyle='--', 
            label='Energy budget', linewidth=2)
plt.xlabel('Neuron (sorted by rate)')
plt.ylabel('Cumulative Energy Cost')
plt.title('Cumulative Metabolic Cost')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:
1. **Ion channels**: Hodgkin-Huxley dynamics for realistic conductances
2. **Multi-compartment neurons**: Cable theory for dendritic integration
3. **Synaptic plasticity**: STDP and STP for learning dynamics
4. **Metabolic constraints**: ATP budgets for biological realism
5. **Foundation model testing**: Applying constraints to AI predictions

These biophysical models provide a rigorous testbed for evaluating whether foundation model representations are consistent with known neural mechanisms.